# MoE: One Model, Many Experts

> **Background**: GPT-4 is rumored to have ~1.8T parameters, but each inference only activates a subset. How?
>
> Goal for this part: **understand the core idea of Mixture of Experts (MoE): replace the FFN layer with multiple experts, and activate only a few per token.**

One-sentence intuition:
**A dense Transformer FFN is one general practitioner. MoE turns it into 8 specialists, and each token visits only the best-matching 2 specialists.**


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

### 1. Recap: the FFN layer in a standard Transformer

Each Transformer block has an FFN (feed-forward network):

```
input x (d_model=512)
  |
Linear(512 -> 2048)   # up-projection
  |
ReLU / GELU
  |
Linear(2048 -> 512)   # down-projection
  |
output
```

Parameter count: `512*2048 + 2048*512 ~= 2M`.

**Problem**: every token goes through the same FFN. Whether it is "the" or "quantum mechanics", it is processed by the same parameters.
To make the model smarter you scale up the FFN -> parameters explode.


### 2. Core idea of MoE: multiple experts + a router

Split one big FFN into N smaller FFNs (experts), and add a router (gate) that chooses which experts each token should use.

```
               +-------------+
               |   Router    |  <- decides who to call
               |   (Gate)    |
               +--+--+--+----+
                  |  |  |
          +-------+  |  +-------+
          v          v          v
     +--------+ +--------+ +--------+
     |Expert 1| |Expert 2| |Expert 3|  ... (8 experts)
     | (FFN)  | | (FFN)  | | (FFN)  |
     +--------+ +--------+ +--------+
          v          v          v
          +----------+----------+
                 weighted sum
```

**Key point**: each token activates only top-k experts (often k=2), not all 8.

Examples:

```
"the"      -> router -> experts 1, 5  (easy token)
"quantum"  -> router -> experts 3, 7  (physics-ish)
"hello"    -> router -> experts 2, 6  (English-ish)
```

**Effect**:
- Total parameters = 8 * (one expert) (large!)
- Compute per inference = 2 * (one expert) (close to dense FFN!)
- **More parameters but not much more compute**: the MoE "magic".


In [2]:
class MoELayer(nn.Module):
    """Mixture-of-Experts (MoE) FFN layer.

    Args:
        d_model: Hidden size.
        num_experts: Number of experts.
        top_k: How many experts to activate per token.
    """
    def __init__(self, d_model, num_experts=8, top_k=2, expert_dim=None):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        expert_dim = expert_dim or 4 * d_model

        # Router: input d_model -> num_experts scores
        self.gate = nn.Linear(d_model, num_experts, bias=False)

        # N experts, each is a small FFN
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, d_model)
            )
            for _ in range(num_experts)
        ])

    def forward(self, x):
        """Forward.

        x: [batch, seq_len, d_model]

        Flow:
        1. Router scores each token.
        2. Pick top-k experts.
        3. Compute outputs only for those k experts.
        4. Weighted sum.
        """
        batch_size, seq_len, d_model = x.shape

        # Step 1: router scores
        gate_logits = self.gate(x)  # [batch, seq_len, num_experts]

        # Step 2: top-k routing
        top_k_logits, top_k_indices = torch.topk(gate_logits, self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)  # normalized weights

        # Step 3 & 4: per token, run selected experts and combine
        output = torch.zeros_like(x)

        for b in range(batch_size):
            for s in range(seq_len):
                token = x[b, s]  # [d_model]
                for k in range(self.top_k):
                    expert_id = top_k_indices[b, s, k].item()
                    weight = top_k_weights[b, s, k]
                    output[b, s] += weight * self.experts[expert_id](token)

        return output


In [3]:
# Show how routing works in a tiny example
moe = MoELayer(d_model=16, num_experts=8, top_k=2)

# Simulate 3 tokens
x = torch.randn(1, 3, 16)

# Router scores for each token
with torch.no_grad():
    gate_scores = moe.gate(x).squeeze(0)  # [3, 8]
    top_k_vals, top_k_idx = torch.topk(gate_scores, 2, dim=-1)

print("=== Router picks experts for 3 tokens ===")
print()
for i in range(3):
    experts = top_k_idx[i].tolist()
    weights = F.softmax(top_k_vals[i], dim=-1).tolist()
    print(f"Token {i}: selected experts {experts}, weights {[f'{w:.2f}' for w in weights]}")

print()
print("Each token activates only 2/8 = 25% of the experts")
print("Total parameters add up across experts, but FLOPs are closer to top-k")


=== Router picks experts for 3 tokens ===

Token 0: selected experts [0, 2], weights ['0.56', '0.44']
Token 1: selected experts [6, 4], weights ['0.50', '0.50']
Token 2: selected experts [0, 5], weights ['0.52', '0.48']

Each token activates only 2/8 = 25% of the experts
Total parameters add up across experts, but FLOPs are closer to top-k


### 2.1 Compare: dense Transformer block vs MoE block

The best way to understand MoE is to print the module structure, not just read formulas.

Dense decoder block mainline:

```text
x -> Attention -> FFN -> output
```

MoE decoder block mainline:

```text
x -> Attention -> Router -> pick top-k FFN experts -> output
```

So MoE usually keeps Attention unchanged, and replaces the FFN with "router + multiple expert FFNs".


In [4]:
# Compare a Dense decoder block vs an MoE decoder block by printing the modules
class TinyDenseBlock(nn.Module):
    """Skeleton of a dense Transformer decoder block: Attention + single FFN."""
    def __init__(self, d_model=16, num_heads=2, ffn_dim=64):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x, need_weights=False)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class TinyMoEBlock(nn.Module):
    """Skeleton of an MoE decoder block: Attention + Router + multiple FFN experts."""
    def __init__(self, d_model=16, num_heads=2, num_experts=4, top_k=2, expert_dim=64):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.moe = MoELayer(d_model, num_experts=num_experts, top_k=top_k, expert_dim=expert_dim)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x, need_weights=False)
        x = self.norm1(x + attn_out)
        moe_out = self.moe(x)
        x = self.norm2(x + moe_out)
        return x


dense_block = TinyDenseBlock()
moe_block = TinyMoEBlock(num_experts=4, top_k=2)

print("=== Dense Transformer Block ===")
print(dense_block)

print()
print("=== MoE Transformer Block ===")
print(moe_block)


=== Dense Transformer Block ===
TinyDenseBlock(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
  )
  (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (ffn): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
  )
  (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
)

=== MoE Transformer Block ===
TinyMoEBlock(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
  )
  (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (moe): MoELayer(
    (gate): Linear(in_features=16, out_features=4, bias=False)
    (experts): ModuleList(
      (0-3): 4 x Sequential(
        (0): Linear(in_features=16, out_features=64, bias=True)
        (1): ReLU()
        (2): Linear(in_features=64, out_features=16, bias=True)
    

From the printed structure, focus on two details:

1. Dense block has a single `ffn`.
2. MoE block has `moe` after `self_attn`, containing a `gate` and multiple `experts`.

This is the structural evidence: **Attention stays; FFN becomes multiple experts.**


In [5]:
# Trace one forward pass: shapes and where routing happens
trace_x = torch.randn(1, 3, 16)

print("=== Dense Block Trace ===")
with torch.no_grad():
    dense_attn_out, _ = dense_block.self_attn(trace_x, trace_x, trace_x, need_weights=False)
    dense_after_attn = dense_block.norm1(trace_x + dense_attn_out)
    dense_ffn_out = dense_block.ffn(dense_after_attn)
    dense_out = dense_block.norm2(dense_after_attn + dense_ffn_out)

print(f"input:        {tuple(trace_x.shape)}")
print(f"attention:    {tuple(dense_attn_out.shape)}")
print(f"single FFN:   {tuple(dense_ffn_out.shape)}")
print(f"output:       {tuple(dense_out.shape)}")

print()
print("=== MoE Block Trace ===")
with torch.no_grad():
    moe_attn_out, _ = moe_block.self_attn(trace_x, trace_x, trace_x, need_weights=False)
    moe_after_attn = moe_block.norm1(trace_x + moe_attn_out)
    gate_logits = moe_block.moe.gate(moe_after_attn)
    top_vals, top_idx = torch.topk(gate_logits, moe_block.moe.top_k, dim=-1)
    moe_out_raw = moe_block.moe(moe_after_attn)
    moe_out = moe_block.norm2(moe_after_attn + moe_out_raw)

print(f"input:        {tuple(trace_x.shape)}")
print(f"attention:    {tuple(moe_attn_out.shape)}")
print(f"gate logits:  {tuple(gate_logits.shape)}  # scores per token per expert")
print(f"top-k index:  {tuple(top_idx.shape)}      # chosen expert ids")
print(f"MoE FFN:      {tuple(moe_out_raw.shape)}")
print(f"output:       {tuple(moe_out.shape)}")
print()
print("Selected experts per token:")
for token_i, experts in enumerate(top_idx[0].tolist()):
    print(f"token {token_i}: experts {experts}")


=== Dense Block Trace ===
input:        (1, 3, 16)
attention:    (1, 3, 16)
single FFN:   (1, 3, 16)
output:       (1, 3, 16)

=== MoE Block Trace ===
input:        (1, 3, 16)
attention:    (1, 3, 16)
gate logits:  (1, 3, 4)  # scores per token per expert
top-k index:  (1, 3, 2)      # chosen expert ids
MoE FFN:      (1, 3, 16)
output:       (1, 3, 16)

Selected experts per token:
token 0: experts [3, 2]
token 1: experts [1, 3]
token 2: experts [3, 2]


### 2.2 Print a real MoE model from HuggingFace

Once you understand the toy skeleton, look at a real decoder layer from engineering code.
Here we do not download weights. We only instantiate a small config (Qwen2-MoE / Mixtral-style) to print module order.

When reading the printout, watch for:

1. `self_attn` still comes first.
2. Where a dense FFN would be, you see `mlp` / `SparseMoeBlock`.
3. Inside MoE you see a `gate` and multiple `experts`.

In newer `transformers`, expert weights may be packed into large tensors (so you may not literally see `expert_0`, `expert_1`, ...).
So also inspect parameter shapes: the 0-th dimension of something like `experts.gate_up_proj` corresponds to the number of experts.


In [6]:
# printreal HuggingFace MoE decoder layer，and trace one router output
import inspect
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

from transformers import Qwen2MoeConfig, Qwen2MoeForCausalLM
from transformers import MixtralConfig, MixtralForCausalLM

qwen_cfg = Qwen2MoeConfig(
    vocab_size=128,
    hidden_size=32,
    intermediate_size=64,
    moe_intermediate_size=64,
    shared_expert_intermediate_size=64,
    num_hidden_layers=1,
    num_attention_heads=4,
    num_key_value_heads=4,
    num_experts=4,
    num_experts_per_tok=2,
)
qwen_moe = Qwen2MoeForCausalLM(qwen_cfg)
qwen_layer = qwen_moe.model.layers[0]

print("=== Qwen2-MoE decoder layer ===")
print(qwen_layer)

print()
print("=== Qwen2-MoE MoE parameter shapes ===")
for name, param in qwen_layer.mlp.named_parameters():
    if name.startswith("gate") or name.startswith("experts") or name.startswith("shared"):
        print(f"{name:32s} {tuple(param.shape)}")

mixtral_cfg = MixtralConfig(
    vocab_size=128,
    hidden_size=32,
    intermediate_size=64,
    num_hidden_layers=1,
    num_attention_heads=4,
    num_key_value_heads=4,
    num_local_experts=4,
    num_experts_per_tok=2,
)
mixtral_moe = MixtralForCausalLM(mixtral_cfg)
mixtral_layer = mixtral_moe.model.layers[0]

print()
print("=== Mixtral decoder layer ===")
print(mixtral_layer)

print()
print("=== Mixtral MoE parameter shapes ===")
for name, param in mixtral_layer.mlp.named_parameters():
    if name.startswith("gate") or name.startswith("experts"):
        print(f"{name:32s} {tuple(param.shape)}")

print()
print("=== Qwen2-MoE layer.forward source excerpt ===")
source = inspect.getsource(type(qwen_layer).forward).splitlines()
for line in source:
    if "self_attn" in line or "mlp" in line or "layernorm" in line:
        print(line)

print()
print("=== Qwen2-MoE mlp.forward source excerpt ===")
source = inspect.getsource(type(qwen_layer.mlp).forward).splitlines()
for line in source:
    if "gate" in line or "expert" in line or "router" in line or "top" in line:
        print(line)

input_ids = torch.randint(0, 128, (1, 6))
with torch.no_grad():
    qwen_out = qwen_moe(input_ids=input_ids, output_router_logits=True)

router_logits = qwen_out.router_logits[0]
top_vals, top_idx = torch.topk(router_logits, qwen_cfg.num_experts_per_tok, dim=-1)

print()
print("=== Qwen2-MoE router trace ===")
print(f"input_ids:      {tuple(input_ids.shape)}")
print(f"logits:         {tuple(qwen_out.logits.shape)}")
print(f"router logits:  {tuple(router_logits.shape)}")
print(f"top-k experts:  {tuple(top_idx.shape)}")
print()
print("Selected experts for the first 6 tokens:")
for token_i, experts in enumerate(top_idx[:6].tolist()):
    print(f"token {token_i}: experts {experts}")


=== Qwen2-MoE decoder layer ===
Qwen2MoeDecoderLayer(
  (self_attn): Qwen2MoeAttention(
    (q_proj): Linear(in_features=32, out_features=32, bias=True)
    (k_proj): Linear(in_features=32, out_features=32, bias=True)
    (v_proj): Linear(in_features=32, out_features=32, bias=True)
    (o_proj): Linear(in_features=32, out_features=32, bias=False)
  )
  (mlp): Qwen2MoeSparseMoeBlock(
    (gate): Qwen2MoeTopKRouter()
    (experts): Qwen2MoeExperts(
      (act_fn): SiLUActivation()
    )
    (shared_expert): Qwen2MoeMLP(
      (gate_proj): Linear(in_features=32, out_features=64, bias=False)
      (up_proj): Linear(in_features=32, out_features=64, bias=False)
      (down_proj): Linear(in_features=64, out_features=32, bias=False)
      (act_fn): SiLUActivation()
    )
    (shared_expert_gate): Linear(in_features=32, out_features=1, bias=False)
  )
  (input_layernorm): Qwen2MoeRMSNorm((32,), eps=1e-06)
  (post_attention_layernorm): Qwen2MoeRMSNorm((32,), eps=1e-06)
)

=== Qwen2-MoE MoE param

### 3. MoE parameters vs compute

This is MoE's most important advantage:

```
Dense model:
  FFN params: 2M
  Compute per forward: all 2M participate

  Scale up -> params and compute grow together

MoE (8 experts, top-2):
  FFN params: 8 * 2M = 16M   (8x parameters!)
  Compute:    2 * 2M = 4M    (only 2x compute)

  Params and compute are decoupled.
```

That is why Mixtral 8x7B can have ~7B-level compute but ~47B "capacity".


### 4. Training challenge: load balancing

MoE has a built-in pitfall: **the router may become lazy and send most tokens to only a few experts.**

```
Bad (imbalanced):
  expert 1: ####################  (overloaded)
  expert 2: ####
  expert 3: #
  expert 4-8: (idle)

Good (balanced):
  expert 1: ######
  expert 2: ######
  ...
  expert 8: ######
```

**Solution**: add an auxiliary loss that penalizes imbalance.

```python
# load-balancing loss (simplified)
# encourage expert assignment to be more even

load_balance_loss = 0
for expert_i in range(num_experts):
    actual_load = count_tokens_assigned_to(expert_i)
    ideal_load = total_tokens * top_k / num_experts
    load_balance_loss += (actual_load - ideal_load) ** 2
```


### 5. Inference challenge: you must load all experts

Even though each token only activates 2 experts, **you don't know which 2 in advance**, so all expert parameters must be resident in GPU memory.

```
Dense 7B:  VRAM ~= 14GB (FP16)
MoE 8x7B:  VRAM ~= 94GB (FP16)   (about 8x)
```

So MoE can be "fast" (less compute) but "heavy" (more memory).

**Common solutions**:
- quantization (INT4/INT8)
- offload cold experts to CPU RAM
- multi-GPU distributed deployment


### 6. Famous MoE models

| Model | Total params | Active params | #Experts | Top-K |
|:---|:---|:---|:---|:---|
| **Mixtral 8x7B** | 47B | 13B | 8 | 2 |
| **DeepSeek-V2** | 236B | 21B | 160 | 6 |
| **GPT-4 (rumor)** | 1.8T | ~280B | 8-16 | 2 |
| **Qwen2.5-MoE** | 57B | 14B | 64 | 8 |

Key number: active / total ~= 10-20%.
So 80-90% of parameters are "sleeping" on each inference.


### 7. Another intuition

```
Dense model = one general practitioner
  - can treat anything
  - but not deeply specialized
  - to get stronger you scale this one doctor -> slower

MoE model = a hospital with 8 specialists
  - cardiology, neurology, orthopedics, ...
  - each patient visits only the best 2 specialists
  - more staff overall (more params), but shorter wait per patient (less compute)

Router = triage nurse
  "headache" -> neurology + internal medicine
  "knee pain" -> orthopedics + rehab
```


### 8. Advanced topics

**DeepSeek-V2 ideas (high level)**:
- classic MoE uses "hard routing" (top-k experts, others get 0)
- DeepSeek uses "soft routing" plus "shared experts"
- shared expert: always-on expert for general knowledge
- routed experts: top-k experts for specialized skills

```
DeepSeek-style MoE:
  token -> shared expert (always) + routed experts (top-k)

  shared expert handles: grammar, common sense
  routed experts handle: math, programming, translation, ...
```

**Expert parallelism**:
- place different experts on different GPUs
- router dispatches tokens to GPUs
- requires all-to-all communication (often the main MoE cost)


---

## MoE Summary

1. [ok] Dense FFN is one "generalist": all tokens share one parameter set
2. [ok] MoE replaces FFN with N "specialists" plus a router
3. [ok] Each token activates only top-k experts -> more params but less compute per token
4. [ok] Params and compute decouple -> more knowledge for similar compute
5. [ok] Training challenge: load balancing -> add auxiliary loss
6. [ok] Inference challenge: all experts must be in VRAM -> quantization / multi-GPU
7. [ok] Famous MoE: Mixtral, DeepSeek-V2, GPT-4 (rumor)
8. [ok] Active params ~= 10-20% of total

One-sentence summary: MoE is many experts in one model; the router chooses which 2 experts each token visits.
